## RNN (LSTM) vs Transformer for Sequence Classification: A Practical Tutorial

### Introduction

Sequential data (such as text or time series) is common in real-world tasks like sentiment analysis, language translation, and forecasting. Two popular neural network architectures for modeling sequence data are Recurrent Neural Networks (RNNs) and Transformers. Recurrent Neural Networks process input sequences one timestep at a time, maintaining an internal state that captures past information. Transformers, on the other hand, process sequences using self-attention mechanisms that allow direct access to any part of the sequence, enabling better capture of long-range dependencies.


Over the past few years, Transformers have revolutionized sequence modeling in NLP, often outperforming RNNs on tasks with long sequences. They can handle long contexts more efficiently by looking at the entire sequence in parallel rather than step-by-step as RNNs do. However, RNNs remain useful for many problems, especially when the sequences are not too long or when computational resources are limited. In this tutorial, we will use moderate computational resources to implement and compare an LSTM-based RNN and a Transformer model on a text classification task (sentiment analysis). We will use PyTorch to build and train both models from scratch on the IMDb movie review dataset and evaluate their performance in terms of accuracy and training efficiency. Finally, we’ll discuss when to use each model based on our findings and general insights from the deep learning community.

What you will learn:
- How to prepare text data for sequence models (tokenization, padding, etc.).
- How to implement an LSTM-based RNN for text classification.
- How to implement a Transformer encoder for text classification.
- How to train and evaluate both models, and compare their performance.
- Key differences, advantages, and limitations of RNNs vs Transformers in practice.

# Dataset and Preprocessing

We will use the IMDb movie review dataset for sentiment classification. This dataset contains 50,000 movie reviews from IMDb, labeled as positive or negative. It’s a classic benchmark for text classification:
- Training set: 25,000 reviews (50% positive, 50% negative).
- Test set: 25,000 reviews (50% positive, 50% negative).

Each review is a sequence of words of varying length (anywhere from a few words to several hundred words). To feed this data into our models, we need to convert the text into numeric form. The typical preprocessing steps are:


- Tokenization: Split each review into a sequence of tokens (words or subwords).
- Vocabulary building: Assign each unique token an integer index. We’ll build a vocabulary of the most frequent words and map rare words to an <UNK> (unknown) token.
- Encoding: Convert each review into a sequence of token indices using the vocabulary.
- Padding/truncation: Pad sequences with a special <PAD> token (index 0) to a fixed length, or truncate longer reviews, so that all sequences in a batch have the same length.
- Train/validation split: We will use part of the training set for validation to tune our models.

For simplicity and moderate resource usage, we will:
- Limit the vocabulary size to the top 10,000 most frequent words.
- Truncate or pad each review to a maximum length of 200 tokens. (This covers most reviews while keeping sequence length manageable.)
- Use a subset of the training data for quicker training (optional). For example, we might use 15,000 reviews for training and 10,000 for validation.

We use the IMDb Large Movie Review Dataset (often just called the IMDb dataset) which contains 50,000 movie reviews split evenly into 25k training and 25k test reviews. Each review is labeled as positive or negative. The data is available from the Stanford AI Lab site as a compressed archive (aclImdb_v1.tar.gz).Download the dataset: You can download and extract it manually, or use shell commands in a Jupyter notebook. For example:

download by go to https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

In [3]:
import os
train_pos_path = "aclImdb/train/pos"
train_neg_path = "aclImdb/train/neg"
print("Number of training positive reviews:", len(os.listdir(train_pos_path)))
print("Number of training negative reviews:", len(os.listdir(train_neg_path)))


Number of training positive reviews: 12500
Number of training negative reviews: 12500


**Data Preprocessing**

Raw text needs to be converted into numeric tensors before feeding to a neural network. We will perform the following preprocessing steps:
- Loading and Tokenization: Read the review text files and split the text into tokens (words). Clean the text by lowercasing and removing punctuation.
- Vocabulary Creation: Build a vocabulary of the most frequent words, mapping each word to an integer index.
- Encoding: Convert each review into a sequence of word indices using the vocabulary (unknown words get a special index).
- Padding/Truncation: Pad or truncate each sequence to a fixed length so they can be batched into tensors.


## Loading and Tokenizing Reviews
We'll gather all training reviews and their labels first. The training data will be used to build the vocabulary. The IMDb dataset comes with train/test split, so we'll keep those separate. To load the reviews:

In [6]:
import re

# Paths to the review files
train_pos_dir = "aclImdb/train/pos"
train_neg_dir = "aclImdb/train/neg"
test_pos_dir  = "aclImdb/test/pos"
test_neg_dir  = "aclImdb/test/neg"

train_texts = []
train_labels = []
test_texts = []
test_labels = []

# Load and label positive training reviews
for fname in os.listdir(train_pos_dir):
    with open(os.path.join(train_pos_dir, fname), 'r', encoding='utf-8') as f:
        text = f.read().strip()
        train_texts.append(text)
        train_labels.append(1)  # positive label 1

# Load and label negative training reviews
for fname in os.listdir(train_neg_dir):
    with open(os.path.join(train_neg_dir, fname), 'r', encoding='utf-8') as f:
        text = f.read().strip()
        train_texts.append(text)
        train_labels.append(0)  # negative label 0

# Likewise for test data
for fname in os.listdir(test_pos_dir):
    with open(os.path.join(test_pos_dir, fname), 'r', encoding='utf-8') as f:
        text = f.read().strip()
        test_texts.append(text)
        test_labels.append(1)
for fname in os.listdir(test_neg_dir):
    with open(os.path.join(test_neg_dir, fname), 'r', encoding='utf-8') as f:
        text = f.read().strip()
        test_texts.append(text)
        test_labels.append(0)

print(f"Loaded {len(train_texts)} training reviews and {len(test_texts)} test reviews.")


Loaded 25000 training reviews and 25000 test reviews.


Next, we define a function to tokenize and clean a review string. We'll remove HTML tags (if any), punctuation, and make everything lowercase. We'll use a simple regex to keep only letters and numbers (you could also use more advanced tokenizers).

In [8]:
def tokenize(text):
    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)
    # Keep only letters and standard punctuation (replace others with space)
    text = re.sub(r"[^a-zA-Z0-9\s]", ' ', text)
    # Lowercase the text
    text = text.lower()
    # Split into tokens by whitespace
    tokens = text.split()
    return tokens

# Tokenize all reviews
train_tokens = [tokenize(review) for review in train_texts]
test_tokens  = [tokenize(review) for review in test_texts]

# Peek at one tokenized example
print(train_texts[0][:100], "->", train_tokens[0][:20])


For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagi -> ['for', 'a', 'movie', 'that', 'gets', 'no', 'respect', 'there', 'sure', 'are', 'a', 'lot', 'of', 'memorable', 'quotes', 'listed', 'for', 'this', 'gem', 'imagine']


## Building the Vocabulary

Now we build a vocabulary (word index mapping) based on the training tokens. It's common to limit the vocabulary size to the top N most frequent words to avoid very rare words. In this dataset, there are around 130k unique tokens if we include everything. Using all unique words can hurt performance (very sparse, many rare words), so we will limit to the most frequent, for example, 10,000 words. (This is a common choice in literature, often 10k or 20k most common words.) Words outside this top list will be treated as "unknown."

In [9]:
from collections import Counter

# Count frequency of each token in the training set
word_counts = Counter(token for review in train_tokens for token in review)
print(f"Total unique tokens in training data: {len(word_counts)}")

# Select most common words for the vocabulary
vocab_size = 10000  # limit vocab to top 10k
most_common = word_counts.most_common(vocab_size - 2)  # -2 to account for special tokens
# We will reserve indices 0 and 1 for special tokens (<PAD> and <UNK>)
word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for i, (word, freq) in enumerate(most_common, start=2):
    word_to_idx[word] = i

print(f"Vocabulary size (including PAD/UNK): {len(word_to_idx)}")

Total unique tokens in training data: 74475
Vocabulary size (including PAD/UNK): 10000


We added two special tokens: <PAD> for padding and <UNK> for any unknown/out-of-vocabulary word. We assigned <PAD> index 0 (we will use 0 for padding in sequences) and <UNK> index 1. Common words like "the", "and", etc., will get low indices since they appear frequent. For example, "the" might be index 2, "and" 3, etc., depending on frequencies. This frequency-based indexing makes it easy to filter out rare words.

## Encoding and Padding Sequences

With the vocabulary in hand, we encode each review’s token list into a sequence of integers. Words not in our word_to_idx (e.g., a rare word not in top 10k) will be mapped to <UNK> (index 1).We also need to pad or truncate each sequence to a fixed length. Neural networks typically process batches of equal-length sequences. We'll pick a maximum sequence length (for example, 250 words). The average IMDb review is around 230 words, and most reviews are under 500 words, so 250 is a reasonable cutoff for this tutorial. Reviews longer than 250 words will be truncated, and shorter reviews will be padded with <PAD> (zeros) at the end. (Padding at the end is common for LSTMs/Transformers; padding at the beginning is another option but either works if handled consistently.)Let's encode and pad.


In [10]:
max_length = 250

def encode_and_pad(tokens):
    # Encode tokens to indices
    indices = [word_to_idx.get(token, 1) for token in tokens]  # 1 is <UNK> for unknown
    # Truncate if longer than max_length
    if len(indices) > max_length:
        indices = indices[:max_length]
    # Pad with 0s if shorter than max_length
    if len(indices) < max_length:
        indices += [0] * (max_length - len(indices))
    return indices

# Encode all training and test sequences
train_sequences = [encode_and_pad(tok_list) for tok_list in train_tokens]
test_sequences  = [encode_and_pad(tok_list) for tok_list in test_tokens]

print("Example encoded review (first 20 indices):", train_sequences[0][:20])
print("Length of encoded review:", len(train_sequences[0]))


Example encoded review (first 20 indices): [16, 4, 18, 12, 213, 58, 1150, 40, 251, 26, 4, 175, 5, 897, 4374, 3575, 16, 11, 1516, 830]
Length of encoded review: 250


Now we have train_sequences and test_sequences which are lists of equal-length lists of integers (each length = 250). Each integer is an index in our vocabulary. For example, if "the" is index 2, you'll see "2" wherever "the" appeared in the text. Unknown words will appear as "1". And padded positions are "0".At this point, our data is ready to be fed into PyTorch models. We just need to wrap it in a PyTorch Dataset and DataLoader for convenient batching.

## Creating PyTorch Dataset and DataLoader

We'll create a custom Dataset to hold our encoded sequences and labels, and then use DataLoader to handle batching and shuffling.

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

class IMDBDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = [torch.tensor(seq, dtype=torch.long) for seq in sequences]
        self.labels    = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

# Create dataset instances
train_dataset = IMDBDataset(train_sequences, train_labels)
test_dataset  = IMDBDataset(test_sequences, test_labels)

# Create DataLoaders for batching
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

We use shuffle=True for training data so that each epoch the model sees batches in a new order (which helps training). We keep shuffle=False for the test loader to evaluate in a fixed order (shuffling isn’t necessary for evaluation).Now we're ready to define our models.

## Defining the Models
We'll implement two models using PyTorch's nn.Module:

1. LSTM-based RNN: An Embedding layer followed by an LSTM (recurrent layer) and a linear layer to output the sentiment class.
2. Transformer Encoder: An Embedding layer (with added positional encoding) followed by Transformer encoder layers and a final linear layer for classification.

Both models will output a prediction for each input review (binary classification: positive or negative). We will use a size-2 output (for classes 0 and 1) and later apply a softmax or use CrossEntropyLoss which expects raw logits of size 2.

## LSTM-based Model
An LSTM RNN processes the review word by word (in sequence order) and retains an internal state. We will use the final hidden state of the LSTM as a summary of the whole review and pass it to a linear layer for classification. Let's define an LSTMClassifier module:

In [12]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)  # embed words, pad idx=0
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, 2)  # output 2 classes (neg, pos)
    def forward(self, x):
        # x shape: (batch, seq_length)
        embeds = self.embedding(x)            # (batch, seq_length, embed_dim)
        output, (h_n, c_n) = self.lstm(embeds)  # h_n shape: (num_layers, batch, hidden_dim)
        # Use the last layer's hidden state for final prediction:
        final_hidden = h_n[-1]               # (batch, hidden_dim)
        final_hidden = self.dropout(final_hidden)
        logits = self.fc(final_hidden)       # (batch, 2)
        return logits


A few notes on this implementation:
- We set padding_idx=0 in the Embedding so that the embedding vector for the PAD token is a zero-vector by default (this ensures padded tokens don’t contribute to the sequence representation).
- batch_first=True in LSTM means our input shape is (batch, seq, feature); this matches the (batch, seq_length, embed_dim) shape of embeds.
- h_n[-1] gives the final hidden state of the last LSTM layer for each sequence in the batch. This is our representation of the whole sequence. We apply dropout to it for regularization, then feed to a fully connected layer to get our two class scores.
- We could also use a bidirectional LSTM or multiple LSTM layers to potentially improve performance, but we keep it single-layer and unidirectional for clarity here.

## Transformer Encoder Model


Transformers use self-attention to process all words in a sequence in parallel (not strictly left-to-right like an RNN). We will implement a Transformer encoder that produces contextualized embeddings for each position, then aggregate those into a single vector for classification. One common approach is to prepend a special [CLS] token and use its output embedding for classification (like BERT does), but for simplicity, we'll instead average the transformer outputs from all positions.One challenge is that Transformers are position-agnostic, so we need to provide positional information. We'll add a positional embedding to the token embeddings.Let's define a TransformerClassifier module:

In [13]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, num_heads=4, num_layers=2, ff_dim=256, max_len=250):
        super(TransformerClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)  # learnable positional embeddings
        # Define a Transformer Encoder layer
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, 
                                                  dim_feedforward=ff_dim, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, 2)
    def forward(self, x):
        # x shape: (batch, seq_length)
        batch_size, seq_len = x.size()
        # Create position indices for each position in the sequence
        pos_indices = torch.arange(0, seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        # Get token embeddings and positional embeddings
        token_embeds = self.embedding(x)            # (batch, seq_len, embed_dim)
        pos_embeds = self.pos_embedding(pos_indices)  # (batch, seq_len, embed_dim)
        # Add token and position embeddings
        x_embedded = token_embeds + pos_embeds       # (batch, seq_len, embed_dim)
        # Create padding mask (True where padding token is present)
        pad_mask = (x == 0)  # shape: (batch, seq_len), True at padded indices
        # Pass through Transformer encoder
        enc_out = self.transformer(x_embedded, src_key_padding_mask=pad_mask)  # (batch, seq_len, embed_dim)
        # Aggregate the encoder outputs; we use mean pooling
        seq_avg = enc_out.mean(dim=1)               # (batch, embed_dim)
        logits = self.fc(seq_avg)                   # (batch, 2)
        return logits


Key points for the Transformer model:

- We use nn.TransformerEncoderLayer and stack two layers (you can adjust num_layers). We set batch_first=True so it expects input shape (batch, seq, embed).
- We use a small number of heads (4) and feed-forward dimension (256) for demonstration. In practice, you might use more heads or larger dimensions.
- We add positional embeddings to the word embeddings to give the model a sense of word order. These are learned embeddings for positions 0 to max_len-1.
- We create a pad_mask where positions with token index 0 (PAD) are marked True, so the Transformer will ignore those positions in its self-attention calculations. This ensures padded tokens don't affect the attention or outputs.
- After the Transformer encoder, we average (mean) the output vectors across the sequence length to get a single vector per sequence. (Alternatively, one could use the output at the first position, use max-pooling, or a designated [CLS] token output. Averaging is a simple way to get a global representation.)
- Then we apply a linear layer to get class logits.

Now we have both models defined.

## Training the Models


We will train and evaluate the LSTM model first, then the Transformer model. Both will be trained from scratch on our dataset. We will use a simple training loop without any high-level libraries.First, set up the training essentials: loss function, optimizer, and device (CPU/GPU). We'll use CrossEntropyLoss since it's a classification task with logits, and Adam optimizer for both models. If a GPU is available, we'll use it for faster training.

In [14]:
import torch.optim as optim

# Select device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Instantiate models
vocab_size = len(word_to_idx)
lstm_model = LSTMClassifier(vocab_size).to(device)
transformer_model = TransformerClassifier(vocab_size).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)
trans_optimizer = optim.Adam(transformer_model.parameters(), lr=0.001)


Using device: cpu


## Training the LSTM Model

In [15]:
num_epochs = 5
for epoch in range(num_epochs):
    lstm_model.train()  # set to training mode
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        lstm_optimizer.zero_grad()
        outputs = lstm_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        lstm_optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, LSTM Training loss: {avg_loss:.4f}")


Epoch 1/5, LSTM Training loss: 0.6934
Epoch 2/5, LSTM Training loss: 0.6962
Epoch 3/5, LSTM Training loss: 0.6919
Epoch 4/5, LSTM Training loss: 0.6833
Epoch 5/5, LSTM Training loss: 0.6711


In [16]:
# Evaluation on test set for LSTM
lstm_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = lstm_model(inputs)              # logits
        preds = torch.argmax(outputs, dim=1)      # predicted class index (0 or 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"LSTM Model Test Accuracy: {accuracy:.4f}")


LSTM Model Test Accuracy: 0.5116


## Training the Transformer Model


In [17]:
num_epochs = 5
for epoch in range(num_epochs):
    transformer_model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        trans_optimizer.zero_grad()
        outputs = transformer_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        trans_optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Transformer Training loss: {avg_loss:.4f}")


Epoch 1/5, Transformer Training loss: 0.5027
Epoch 2/5, Transformer Training loss: 0.3227
Epoch 3/5, Transformer Training loss: 0.2489
Epoch 4/5, Transformer Training loss: 0.1911
Epoch 5/5, Transformer Training loss: 0.1416


In [18]:
transformer_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = transformer_model(inputs)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"Transformer Model Test Accuracy: {accuracy:.4f}")


/Users/yuwang/Dropbox/UO/UO_course/Data Mining/notebook/dm_winter2025/lib/python3.9/site-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Transformer Model Test Accuracy: 0.8372


## Step 1: Positional Encoding


Positional encoding injects information about the position of each token in the sequence. Since Transformers don't have a built-in sense of order (unlike RNNs), we use sinusoidal functions to create unique positional embeddings.

In [19]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        """
        Implements positional encoding as described in Vaswani et al., 2017.
        The sinusoidal encoding helps the model understand token positions.
        
        Args:
        - d_model (int): Embedding dimension.
        - max_len (int): Maximum sequence length.
        - dropout (float): Dropout rate for regularization.
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create a matrix to store positional encodings: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # Shape: (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices and cosine to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)  # Apply sine to even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # Apply cosine to odd indices
        
        # Add batch dimension (1, max_len, d_model)
        pe = pe.unsqueeze(0)  
        
        # Register as buffer so it is saved with the model but does not require gradients
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Add positional encoding to input embeddings.
        
        Args:
        - x (Tensor): Input embeddings of shape (batch_size, seq_len, d_model)
        
        Returns:
        - Tensor of same shape with positional encoding added.
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


## Step 2: Multi-Head Self-Attention Mechanism

Self-attention allows the model to focus on different parts of the input sequence when making predictions. We implement multi-head self-attention, where multiple attention layers run in parallel.

In [20]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        """
        Implements multi-head self-attention.
        
        Args:
        - d_model (int): Embedding size.
        - num_heads (int): Number of attention heads.
        """
        super(MultiHeadSelfAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Each head has dimension d_k
        
        # Linear layers for projecting input into Q, K, V spaces
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)  # Final output layer

    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, mask=None) -> torch.Tensor:
        """
        Compute multi-head self-attention.
        
        Args:
        - Q, K, V (Tensor): Input tensors of shape (batch, seq_len, d_model)
        - mask (Tensor, optional): Mask for padded tokens.
        
        Returns:
        - Output tensor of shape (batch, seq_len, d_model)
        """
        batch_size, seq_len, _ = Q.size()

        # Linear projections for Q, K, V
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)

        # Reshape for multi-heads: (batch_size, num_heads, seq_len, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # Compute scaled dot-product attention
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # Shape: (batch, num_heads, seq_len, seq_len)
        
        # Apply mask if provided
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)  # Shape: (batch, num_heads, seq_len, d_k)

        # Concatenate heads and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(attn_output)


## Step 3: Transformer Encoder Block

Each encoder block consists of:

- Multi-head self-attention
- Feed-forward neural network
- Layer normalization
- Residual connections

In [21]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dim_feedforward: int = 2048, dropout: float = 0.1):
        """
        Implements a single Transformer encoder block.
        
        Args:
        - d_model (int): Embedding size.
        - num_heads (int): Number of attention heads.
        - dim_feedforward (int): Hidden size of the feed-forward layer.
        - dropout (float): Dropout rate.
        """
        super(TransformerEncoderBlock, self).__init__()
        self.attention = MultiHeadSelfAttention(d_model, num_heads)
        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)
        
        # Position-wise feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, d_model)
        )
        
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        """
        Forward pass of the encoder block.
        
        Args:
        - x (Tensor): Input tensor of shape (batch, seq_len, d_model)
        - mask (Tensor, optional): Mask for padded tokens.
        
        Returns:
        - Tensor of shape (batch, seq_len, d_model)
        """
        attn_out = self.attention(x, x, x, mask=mask)
        attn_out = self.dropout1(attn_out)
        x = self.layernorm1(x + attn_out)  # Residual connection + normalization
        
        ff_out = self.ffn(x)
        ff_out = self.dropout2(ff_out)
        return self.layernorm2(x + ff_out)  # Another residual connection + normalization


## Step 4: Full Transformer Model for Sentiment Classification

Now, we stack multiple encoder blocks and add a classification head.

In [23]:
class SentimentTransformer(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 128, num_heads: int = 4, 
                 num_layers: int = 2, dim_feedforward: int = 256, max_seq_len: int = 200, num_classes: int = 2):
        """
        Full Transformer model for sentiment classification.
        """
        super(SentimentTransformer, self).__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len)
        self.encoder_layers = nn.ModuleList([TransformerEncoderBlock(d_model, num_heads, dim_feedforward) for _ in range(num_layers)])
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        x = self.embed(x) * math.sqrt(self.embed.embedding_dim)
        x = self.pos_encoding(x)
        for layer in self.encoder_layers:
            x = layer(x, mask)
        x = x.mean(dim=1)  # Average pooling
        return self.classifier(x)  # Logits


In [26]:
import torch.optim as optim

# Select device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Instantiate models
vocab_size = len(word_to_idx)
MAX_LEN = 250  # Ensure this is consistent across dataset preprocessing

# Update model instantiation
transformer_model = SentimentTransformer(
    vocab_size, d_model=128, num_heads=4, num_layers=2, dim_feedforward=256, 
    max_seq_len=MAX_LEN, num_classes=2
).to(device)


# Loss and optimizer
criterion = nn.CrossEntropyLoss()
trans_optimizer = optim.Adam(transformer_model.parameters(), lr=0.001)


Using device: cpu


In [27]:
num_epochs = 5
for epoch in range(num_epochs):
    transformer_model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        trans_optimizer.zero_grad()
        outputs = transformer_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        trans_optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Transformer Training loss: {avg_loss:.4f}")


Epoch 1/5, Transformer Training loss: 0.5109
Epoch 2/5, Transformer Training loss: 0.4046
Epoch 3/5, Transformer Training loss: 0.3738
Epoch 4/5, Transformer Training loss: 0.3510
Epoch 5/5, Transformer Training loss: 0.3323


In [28]:
transformer_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = transformer_model(inputs)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"Transformer Model Test Accuracy: {accuracy:.4f}")


Transformer Model Test Accuracy: 0.8344
